## Building GP Models from Scratch
Sometimes it is useful to build GP models outside the context of BO for data
visualization and senativity measurements, ie. learned hyperparameters. Here we
demonstrate how to build models from data outside of generators.

For this we use the 3D rosenbrock function test function.

In [ ]:
# set values if testing
import os

from xopt import Xopt, Evaluator
from xopt.generators import RandomGenerator
from xopt.resources.test_functions.rosenbrock import (
    make_rosenbrock_vocs,
)

from xopt.generators.bayesian.visualize import visualize_model
from xopt.generators.bayesian.models import StandardModelConstructor

SMOKE_TEST = os.environ.get("SMOKE_TEST")

# make rosenbrock function vocs in 3D
vocs = make_rosenbrock_vocs(3)
vocs.observables = ["z"]


def evaluate_func(X):
    return {
        "y": X["x0"] ** 2 + X["x1"] ** 2,
        "z": X["x0"] ** 2,
    }


# collect some data using random sampling
evaluator = Evaluator(function=evaluate_func)
generator = RandomGenerator(vocs=vocs)
X = Xopt(generator=generator, evaluator=evaluator)
X.random_evaluate(10)

## Create GP model based on the data

In [ ]:
data = X.data

In [ ]:
model_constructor = StandardModelConstructor(saas_outputs=["z", "y"])

# here we build a model from info (more flexible)
model = model_constructor.build_model(
    input_names=vocs.variable_names,
    outcome_names=["y", "z"],
    data=data,
)

In [ ]:
objective_model = model.models[vocs.output_names.index("y")]

# print raw hyperparameter values
for name, val in objective_model.named_parameters():
    print(name, val)

## Visualize model predictions

In [ ]:
fig, ax = visualize_model(
    model,
    vocs,
    data,
    variable_names=["x0", "x1"],
    reference_point=data.iloc[-1][vocs.variable_names].to_dict(),
)

In [ ]:
data.iloc[-1][vocs.variable_names].to_dict()